# 과제 10 · 01 · ML 예측은 사실이 아니다 — EventCard 와 판정 기준

## 현업 시나리오
설비 MC-01 의 고장 예측 모델이 "열 방산 고장(HDF) 확률 0.45" 라는 이상 이벤트 카드를 보냈습니다. Agent 는 이 카드를 입력으로 받습니다. 그런데 예측은 확률일 뿐입니다. 정비 매뉴얼에는 센서 값으로 고장 유형을 판정하는 기준이 따로 있고, 둘이 어긋나면 사람이 봐야 합니다.

## 학습 목표
- Pydantic 계약으로 EventCard 를 받고, 계약 밖의 값(정답 필드, 없는 유형)을 막는다.
- 매뉴얼 판정 기준을 센서 값에 적용하는 함수를 만든다.
- 예측 유형과 판정 기준이 어긋나는 카드를 구별한다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · Pydantic 계약으로 카드 받기

In [ ]:
# EventCard 는 ML 이 만든 결과입니다. Agent 는 이 모양만 믿고 받습니다.
# Literal 은 허용값 목록입니다. "RNF" 처럼 모델이 예측하지 않는 유형은 들어올 수 없습니다.
# 관찰 포인트: extra="forbid" 라서 계약에 없는 필드(정답 같은 것)가 섞이면 받지 않습니다.
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError

FailureType = Literal["TWF", "HDF", "PWF", "OSF"]

class PracticeStrict(BaseModel):
    model_config = ConfigDict(extra="forbid")

class PracticeSensors(PracticeStrict):
    air_temperature_k: float
    process_temperature_k: float
    rotational_speed_rpm: float
    torque_nm: float
    tool_wear_min: float

class PracticePrediction(PracticeStrict):
    predicted_failure_type: FailureType
    probabilities: dict[FailureType, float]
    confidence_level: Literal["high", "medium", "low"]

class PracticeCard(PracticeStrict):
    event_id: str = Field(pattern=r"^EVT-2025-\d{4}$")
    quality_grade: Literal["L", "M", "H"]
    sensor_snapshot: PracticeSensors
    prediction: PracticePrediction

practice_raw_ok = {
    "event_id": "EVT-2025-0034", "quality_grade": "L",
    "sensor_snapshot": {"air_temperature_k": 301.4, "process_temperature_k": 309.9,
                        "rotational_speed_rpm": 1259, "torque_nm": 63.9, "tool_wear_min": 20},
    "prediction": {"predicted_failure_type": "HDF", "confidence_level": "low",
                   "probabilities": {"TWF": 0.01, "HDF": 0.45, "PWF": 0.02, "OSF": 0.03}},
}
practice_card_ok = PracticeCard.model_validate(practice_raw_ok)
print(practice_card_ok.event_id, "· 예측", practice_card_ok.prediction.predicted_failure_type,
      "· 확률", practice_card_ok.prediction.probabilities["HDF"])

# 정답 필드가 섞이고 없는 유형을 예측한 카드입니다.
practice_raw_bad = {**practice_raw_ok, "machine_failure": True,
                    "prediction": {**practice_raw_ok["prediction"], "predicted_failure_type": "RNF"}}
try:
    PracticeCard.model_validate(practice_raw_bad)
    practice_rejected = []
except ValidationError as error:
    practice_rejected = [".".join(map(str, e["loc"])) for e in error.errors()]
print("받지 않은 곳:", practice_rejected)

**결과 해설** — 계약에 맞는 카드는 그대로 객체가 되고, 정답 필드(`machine_failure`)가 섞이거나 없는 유형(`RNF`)을 예측한 카드는 두 곳 모두 이름이 찍혀 거부됩니다. **Agent 가 정답을 보고 답하면 평가가 성립하지 않으므로** 계약이 그것을 입구에서 막습니다.

### 2단계 · 판정 기준을 함수로

In [ ]:
# 매뉴얼(MC01-MM 부록 A)의 판정 기준을 센서 값에 그대로 적용합니다. 모델의 확률은 보지 않습니다.
# 온도 차·기계 출력·부하 지수는 센서 값에서 계산합니다.
# 관찰 포인트: 네 유형을 모두 계산합니다. 예측 유형 하나만 보면 다른 유형이 성립한 것을 놓칩니다.
PRACTICE_OSF_LIMIT = {"L": 11000, "M": 12000, "H": 13000}

def practice_criteria(sensors, grade):
    temp_diff = sensors.process_temperature_k - sensors.air_temperature_k
    power = sensors.torque_nm * sensors.rotational_speed_rpm * 2 * 3.141592653589793 / 60
    load = sensors.tool_wear_min * sensors.torque_nm
    return {
        "TWF": sensors.tool_wear_min >= 200,
        "HDF": temp_diff < 8.6 and sensors.rotational_speed_rpm < 1380,
        "PWF": power < 3500 or power > 9000,
        "OSF": load > PRACTICE_OSF_LIMIT[grade],
    }

practice_met_ok = practice_criteria(practice_card_ok.sensor_snapshot, practice_card_ok.quality_grade)
for practice_kind, practice_met in practice_met_ok.items():
    print(f"  {practice_kind}: {'해당' if practice_met else '해당 없음'}")

**결과 해설** — 온도 차 8.5 K(< 8.6), 회전수 1,259 rpm(< 1,380)이라 HDF 만 성립합니다. 기계 출력은 약 8,425 W 로 PWF 범위 안이고, 공구 마모 20분·부하 지수 1,278 은 TWF·OSF 와 거리가 멉니다. 예측(HDF)과 판정이 같으므로 이 카드는 근거를 붙여 초안을 쓸 수 있는 경우입니다.

### 3단계 · 예측과 기준이 어긋날 때

In [ ]:
# 같은 계약으로 받은 다른 카드입니다. 모델은 과부하 고장(OSF)을 높은 확률로 예측했습니다.
# 판정은 앞 단계 함수를 그대로 씁니다. 카드마다 규칙을 바꾸지 않습니다.
# 관찰 포인트: 확률이 높다고 판정 기준이 성립하는 것은 아닙니다. 둘이 다르면 사람이 봐야 합니다.
practice_mismatch = PracticeCard.model_validate({
    "event_id": "EVT-2025-0106", "quality_grade": "L",
    "sensor_snapshot": {"air_temperature_k": 300.1, "process_temperature_k": 310.4,
                        "rotational_speed_rpm": 1420, "torque_nm": 48.0, "tool_wear_min": 214},
    "prediction": {"predicted_failure_type": "OSF", "confidence_level": "high",
                   "probabilities": {"TWF": 0.31, "HDF": 0.01, "PWF": 0.02, "OSF": 0.62}},
})
practice_met_mismatch = [k for k, v in practice_criteria(
    practice_mismatch.sensor_snapshot, practice_mismatch.quality_grade).items() if v]
practice_predicted = practice_mismatch.prediction.predicted_failure_type
print("예측:", practice_predicted, "· 기준 성립:", practice_met_mismatch)
print("부하 지수:", practice_mismatch.sensor_snapshot.tool_wear_min * practice_mismatch.sensor_snapshot.torque_nm,
      "/ 한계", PRACTICE_OSF_LIMIT["L"])
print("판단:", "근거로 초안" if practice_predicted in practice_met_mismatch else "예측과 판정이 다름 → 사람에게 넘김(ESC-3)")

assert practice_card_ok.prediction.predicted_failure_type == "HDF"
assert "machine_failure" in practice_rejected, "정답 필드가 계약을 통과했습니다"
assert "prediction.predicted_failure_type" in practice_rejected, "없는 유형이 계약을 통과했습니다"
assert [k for k, v in practice_met_ok.items() if v] == ["HDF"]
assert practice_met_mismatch == ["TWF"] and practice_predicted not in practice_met_mismatch

**결과 해설** — 모델은 OSF 확률 0.62(high)로 예측했지만 부하 지수는 214 × 48 = 10,272 로 L 등급 한계 11,000 에 못 미칩니다. 대신 공구 마모 214분이 TWF 위험 구간(200분 이상)에 들어 있습니다. **확신도가 높은 예측도 판정 기준과 어긋날 수 있고**, SOP 는 이것을 ESC-3(예측과 판정 불일치)으로 사람에게 넘기라고 정합니다.

## 직접 해 볼 과제
토크를 바꿔 보세요. 몇 Nm 부터 OSF 기준이 성립합니까? 그때 판단은 어떻게 바뀝니까? (힌트: 부하 지수 = 공구 마모 시간 × 토크, L 등급 한계 11,000)

In [ ]:
# practice_try_torque 를 바꿔 다시 실행하세요. 예: 51.0, 52.0, 60.0
# 판정 함수와 카드 계약은 그대로 둡니다. 센서 값 하나만 바꿉니다.
practice_try_torque = 48.0
practice_try = practice_mismatch.sensor_snapshot.model_copy(update={"torque_nm": practice_try_torque})
practice_try_met = [k for k, v in practice_criteria(practice_try, "L").items() if v]
print(f"토크 {practice_try_torque} Nm → 부하 지수 {practice_try.tool_wear_min * practice_try_torque:,.0f} · 기준 성립 {practice_try_met}")

## 중간 결과
계약이 거부한 두 곳(정답 필드, 없는 유형), 첫 카드의 유형별 판정, 그리고 둘째 카드에서 예측(OSF)과 판정(TWF)이 갈리는 것을 확인합니다.

## 실패 경계
판정 기준을 예측 유형 하나에만 적용하면 둘째 카드는 "OSF 해당 없음"으로 끝나고, 실제로 위험한 공구 마모(TWF)를 놓칩니다. 그리고 TWF 는 위험 구간에서 무작위로 일어나므로 기준이 성립해도 고장이 아닐 수 있습니다. 판정 기준은 **근거**이지 정답이 아닙니다.

## 실제 app 연결
과제 10 App 의 `domain.py::EventCard` 가 같은 계약(정답 없음, extra 금지)이고, `criteria.py::check_criteria` 가 네 유형을 모두 계산해 매뉴얼 절 번호와 함께 돌려줍니다. 둘이 어긋나면 `routing.py::decide_route` 가 ESC-3 으로 escalation 합니다.

## 다음 Notebook 연결
다음 `02_section_chunking.ipynb` 에서는 판정의 근거가 될 매뉴얼을 절 단위로 자릅니다.